# 온톨로지 작업 노트북

이 노트북은 온톨로지 설계 및 작업을 위한 실습 공간입니다.

**실행 세션:** 터미널에서 `source ./ontology_env.sh` (또는 `source ./ontology_env/bin/activate`) 후 커널을 **ontology_env**로 선택하면 이 환경에서 실행됩니다.

## 1. 라이브러리 설치 및 임포트

In [ ]:
# 필수 라이브러리 설치 (최초 1회만)
#!pip install rdflib owlready2 pandas

In [ ]:
#!pip install rdflib owlready2 pandas

In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL
import pandas as pd
import os 

# Check repository
os.getcwd()

'/Users/dk/Desktop/file/ontology/notebooks'

## 2. 온톨로지 생성 (TBox) - 조선소 최적화 기반

**데이터 소스**: NPS 3BAY 최적화 정의서 기반 데이터
**핵심 엔티티**: PROJ_NO (호선), BLK_NO (블록), WSTG_CODE (송선)

In [9]:
# 그래프 생성
g = Graph()

# 네임스페이스 정의
ex = Namespace("http://example.org/shipyard-optim-onto#")
g.bind("ex", ex)

print("Graph created with namespace:", ex)
print("온톨로지 IRI:", ex)

Graph created with namespace: http://example.org/shipyard-optim-onto#
온톨로지 IRI: http://example.org/shipyard-optim-onto#


- 클래스 정의 (TBox)

In [13]:
# PROJ_NO 

g.add((ex.Project, RDF.type, OWL.Class))

print("클래스 정의완료")

for cls in g.subjects(RDF.type, OWL.Class):
    print(f"{cls}")

클래스 정의완료
http://example.org/shipyard-optim-onto#Project


In [ ]:
# 클래스 정의 (TBox)
# 조선소 최적화 도메인의 핵심 클래스

# 마스터 엔터티
# 날짜 관련 처리는 다음과 같이 설계할 수 있어.
# 
# 1) 날짜 자체를 클래스로 정의: 예시로 Date 라는 클래스를 만들고, TB_CLANDER에 있는 각 날짜(날짜값이 프라이머리 키 역할)를 인스턴스로 생성해.
# 
# 2) 작업일 속성 부여: 해당 Date 인스턴스에 대해 isWorkingDay라는 데이터 프로퍼티(예: boolean 타입)를 추가해서, TB_CLANDER의 '근무일 여부' 컬럼과 매핑.
#    예) ex.isWorkingDay (domain: ex.Date, range: xsd:boolean)
#
# 3) 날짜와 엔티티 연결: Block이나 MoveJob 등 날짜와 연관된 클래스가 있으면, 예를 들어 hasPlannedDate 같은 오브젝트 프로퍼티로 Date 인스턴스와 연결할 수 있어.
#
# 정리하면,
# - Date(Class)와 해당 날짜별 인스턴스
# - isWorkingDay(DataProperty, boolean)
# - 필요하면 작업과 날짜를 연결하는 ObjectProperty(hasPlannedDate 등)
# 이런 식으로 TB_CLANDER 및 작업날짜를 온톨로지에 녹여낼 수 있어.
g.add((ex.Project, RDF.type, OWL.Class))           # "호선"의 추상 클래스 (PROJ_NO 컬럼 기반)
g.add((ex.Block, RDF.type, OWL.Class))             # "블록"의 추상 클래스 (BLK_NO 컬럼 기반)
g.add((ex.WorkingStage, RDF.type, OWL.Class))      # "송선 단계"의 추상 클래스 (WSTG_CODE 컬럼 기반)
g.add((ex.OptimizationPlan, RDF.type, OWL.Class))  # (지금은 데이터가 없지만, 추후 활용할 최적화 계획 클래스)

# 의사결정 단위
g.add((ex.MoveJob, RDF.type, OWL.Class))  # 배정 작업 (PROJ_NO-BLK_NO-WSTG_CODE 조합)

print("클래스 정의 완료:")
for cls in g.subjects(RDF.type, OWL.Class):
    print(f"  - {cls}")

In [ ]:
# Object Properties 정의 (관계)

# 호선-블록 관계
belongsToProject = ex.belongsToProject
g.add((belongsToProject, RDF.type, OWL.ObjectProperty))
g.add((belongsToProject, RDFS.domain, ex.Block))
g.add((belongsToProject, RDFS.range, ex.Project))
g.add((belongsToProject, RDFS.label, Literal("belongsToProject", lang="en")))
g.add((belongsToProject, RDFS.comment, Literal("블록이 속한 호선", lang="ko")))

# 작업의 구성요소
jobProject = ex.jobProject
g.add((jobProject, RDF.type, OWL.ObjectProperty))
g.add((jobProject, RDFS.domain, ex.MoveJob))
g.add((jobProject, RDFS.range, ex.Project))

jobBlock = ex.jobBlock
g.add((jobBlock, RDF.type, OWL.ObjectProperty))
g.add((jobBlock, RDFS.domain, ex.MoveJob))
g.add((jobBlock, RDFS.range, ex.Block))

jobStage = ex.jobStage
g.add((jobStage, RDF.type, OWL.ObjectProperty))
g.add((jobStage, RDFS.domain, ex.MoveJob))
g.add((jobStage, RDFS.range, ex.WorkingStage))

# 블록-송선 관계
hasStage = ex.hasStage
g.add((hasStage, RDF.type, OWL.ObjectProperty))
g.add((hasStage, RDFS.domain, ex.Block))
g.add((hasStage, RDFS.range, ex.WorkingStage))

# 최적화 계획 관계
belongsToPlan = ex.belongsToPlan
g.add((belongsToPlan, RDF.type, OWL.ObjectProperty))
g.add((belongsToPlan, RDFS.domain, ex.Block))
g.add((belongsToPlan, RDFS.range, ex.OptimizationPlan))

print("Object Properties 정의 완료:")
for prop in g.subjects(RDF.type, OWL.ObjectProperty):
    print(f"  - {prop}")

In [ ]:
# 데이터 분석: 고유값 확인
print("고유한 호선 수:", df_block['PROJ_NO'].nunique())
print("고유한 블록 수:", df_block['BLK_NO'].nunique())
print("고유한 송선 코드 수:", df_block['WSTG_CODE'].nunique())
print("\n고유한 송선 코드 목록:")
print(df_block['WSTG_CODE'].unique())
print("\n호선별 블록 수:")
print(df_block.groupby('PROJ_NO')['BLK_NO'].nunique().head(10))

In [ ]:
# ABox 생성: 마스터 데이터 (Project, WorkingStage)

# 고유한 호선(Project) 생성
unique_projects = df_block['PROJ_NO'].unique()
for proj_no in unique_projects:
    project_uri = URIRef(f"{ex}Project_{proj_no}")
    g.add((project_uri, RDF.type, ex.Project))
    g.add((project_uri, projNo, Literal(str(proj_no))))

print(f"생성된 Project 개수: {len(unique_projects)}")

# 고유한 송선(WorkingStage) 생성
unique_stages = df_block['WSTG_CODE'].unique()
for wstg_code in unique_stages:
    stage_uri = URIRef(f"{ex}Stage_{wstg_code}")
    g.add((stage_uri, RDF.type, ex.WorkingStage))
    g.add((stage_uri, wstgCode, Literal(str(wstg_code))))

print(f"생성된 WorkingStage 개수: {len(unique_stages)}")

# 최적화 계획 생성
optz_pln_id = df_block['OPTZ_PLN_ID'].iloc[0]  # 첫 번째 계획 ID 사용
plan_uri = URIRef(f"{ex}Plan_{optz_pln_id}")
g.add((plan_uri, RDF.type, ex.OptimizationPlan))
g.add((plan_uri, optzPlnId, Literal(str(optz_pln_id))))

print(f"생성된 OptimizationPlan: {optz_pln_id}")

In [ ]:
# ABox 생성: Block 및 관계

for idx, row in df_block.iterrows():
    proj_no = str(row['PROJ_NO'])
    blk_no = str(row['BLK_NO'])
    wstg_code = str(row['WSTG_CODE'])
    
    # Block 인스턴스 생성
    block_uri = URIRef(f"{ex}Block_{proj_no}_{blk_no}")
    g.add((block_uri, RDF.type, ex.Block))
    g.add((block_uri, blkNo, Literal(blk_no)))
    
    # 물리적 속성
    if pd.notna(row['BLK_LNTH']):
        g.add((block_uri, blockLength, Literal(float(row['BLK_LNTH']), datatype=XSD.decimal)))
    if pd.notna(row['BLK_BDTH']):
        g.add((block_uri, blockBreadth, Literal(float(row['BLK_BDTH']), datatype=XSD.decimal)))
    if pd.notna(row['PRIORITY']):
        g.add((block_uri, priority, Literal(int(row['PRIORITY']), datatype=XSD.integer)))
    if pd.notna(row['LEAD_TIME']):
        g.add((block_uri, leadTime, Literal(int(row['LEAD_TIME']), datatype=XSD.integer)))
    
    # 관계 설정
    project_uri = URIRef(f"{ex}Project_{proj_no}")
    stage_uri = URIRef(f"{ex}Stage_{wstg_code}")
    plan_uri = URIRef(f"{ex}Plan_{optz_pln_id}")
    
    g.add((block_uri, belongsToProject, project_uri))
    g.add((block_uri, hasStage, stage_uri))
    g.add((block_uri, belongsToPlan, plan_uri))
    
    # MoveJob 생성 (PROJ_NO-BLK_NO-WSTG_CODE 조합)
    job_uri = URIRef(f"{ex}Job_{proj_no}_{blk_no}_{wstg_code}")
    g.add((job_uri, RDF.type, ex.MoveJob))
    g.add((job_uri, jobProject, project_uri))
    g.add((job_uri, jobBlock, block_uri))
    g.add((job_uri, jobStage, stage_uri))

print(f"생성된 Block 개수: {len(df_block)}")
print(f"생성된 MoveJob 개수: {len(df_block)}")

In [ ]:
# 생성된 트리플 수 확인
print(f"총 트리플 수: {len(g)}")
print(f"\n클래스별 인스턴스 수:")
print(f"  Project: {len(list(g.subjects(RDF.type, ex.Project)))}")
print(f"  Block: {len(list(g.subjects(RDF.type, ex.Block)))}")
print(f"  WorkingStage: {len(list(g.subjects(RDF.type, ex.WorkingStage)))}")
print(f"  MoveJob: {len(list(g.subjects(RDF.type, ex.MoveJob)))}")
print(f"  OptimizationPlan: {len(list(g.subjects(RDF.type, ex.OptimizationPlan)))}")

In [ ]:
# 쿼리 2: 특정 호선의 블록 목록
query2 = """
PREFIX ex: <http://example.org/shipyard-optim-onto#>

SELECT ?block ?blkNo ?length ?breadth ?priority
WHERE {
    ?block a ex:Block .
    ?block ex:blkNo ?blkNo .
    ?block ex:belongsToProject ex:Project_2579 .
    OPTIONAL { ?block ex:blockLength ?length }
    OPTIONAL { ?block ex:blockBreadth ?breadth }
    OPTIONAL { ?block ex:priority ?priority }
}
ORDER BY ?blkNo
LIMIT 10
"""

results2 = g.query(query2)
print("=== 호선 2579의 블록 목록 ===")
for row in results2:
    print(f"  블록 {row['blkNo']}: 길이={row.get('length', 'N/A')}, 폭={row.get('breadth', 'N/A')}, 우선순위={row.get('priority', 'N/A')}")

In [ ]:
# 쿼리 3: 송선 코드별 블록 수
query3 = """
PREFIX ex: <http://example.org/shipyard-optim-onto#>

SELECT ?stage ?wstgCode (COUNT(?block) AS ?blockCount)
WHERE {
    ?stage a ex:WorkingStage .
    ?stage ex:wstgCode ?wstgCode .
    ?block a ex:Block .
    ?block ex:hasStage ?stage .
}
GROUP BY ?stage ?wstgCode
ORDER BY DESC(?blockCount)
"""

results3 = g.query(query3)
print("=== 송선 코드별 블록 수 ===")
for row in results3:
    print(f"  {row['wstgCode']}: {row['blockCount']}개 블록")

In [ ]:
# 쿼리 4: MoveJob과 관련된 모든 정보 (JOIN)
query4 = """
PREFIX ex: <http://example.org/shipyard-optim-onto#>

SELECT ?job ?projNo ?blkNo ?wstgCode ?length ?breadth
WHERE {
    ?job a ex:MoveJob .
    ?job ex:jobProject ?project .
    ?project ex:projNo ?projNo .
    ?job ex:jobBlock ?block .
    ?block ex:blkNo ?blkNo .
    ?job ex:jobStage ?stage .
    ?stage ex:wstgCode ?wstgCode .
    OPTIONAL { ?block ex:blockLength ?length }
    OPTIONAL { ?block ex:blockBreadth ?breadth }
}
LIMIT 5
"""

results4 = g.query(query4)
print("=== MoveJob 정보 (상위 5개) ===")
for row in results4:
    print(f"  작업: 호선={row['projNo']}, 블록={row['blkNo']}, 송선={row['wstgCode']}, 크기={row.get('length', 'N/A')}x{row.get('breadth', 'N/A')}")

In [ ]:
# Data Properties 정의 (속성)

# Project 속성
projNo = ex.projNo
g.add((projNo, RDF.type, OWL.DatatypeProperty))
g.add((projNo, RDFS.domain, ex.Project))
g.add((projNo, RDFS.range, XSD.string))

# Block 속성
blkNo = ex.blkNo
g.add((blkNo, RDF.type, OWL.DatatypeProperty))
g.add((blkNo, RDFS.domain, ex.Block))
g.add((blkNo, RDFS.range, XSD.string))

blockLength = ex.blockLength
g.add((blockLength, RDF.type, OWL.DatatypeProperty))
g.add((blockLength, RDFS.domain, ex.Block))
g.add((blockLength, RDFS.range, XSD.decimal))

blockBreadth = ex.blockBreadth
g.add((blockBreadth, RDF.type, OWL.DatatypeProperty))
g.add((blockBreadth, RDFS.domain, ex.Block))
g.add((blockBreadth, RDFS.range, XSD.decimal))

priority = ex.priority
g.add((priority, RDF.type, OWL.DatatypeProperty))
g.add((priority, RDFS.domain, ex.Block))
g.add((priority, RDFS.range, XSD.integer))

leadTime = ex.leadTime
g.add((leadTime, RDF.type, OWL.DatatypeProperty))
g.add((leadTime, RDFS.domain, ex.Block))
g.add((leadTime, RDFS.range, XSD.integer))

# WorkingStage 속성
wstgCode = ex.wstgCode
g.add((wstgCode, RDF.type, OWL.DatatypeProperty))
g.add((wstgCode, RDFS.domain, ex.WorkingStage))
g.add((wstgCode, RDFS.range, XSD.string))

# OptimizationPlan 속성
optzPlnId = ex.optzPlnId
g.add((optzPlnId, RDF.type, OWL.DatatypeProperty))
g.add((optzPlnId, RDFS.domain, ex.OptimizationPlan))
g.add((optzPlnId, RDFS.range, XSD.string))

print("Data Properties 정의 완료:")
for prop in g.subjects(RDF.type, OWL.DatatypeProperty):
    print(f"  - {prop}")

In [ ]:
# XSD 네임스페이스 추가
from rdflib.namespace import XSD

# Restriction 설정 (Cardinality)
# Block은 정확히 1개의 Project에 속함
from rdflib import BNode

restriction1 = BNode()
g.add((ex.Block, RDFS.subClassOf, restriction1))
g.add((restriction1, RDF.type, OWL.Restriction))
g.add((restriction1, OWL.onProperty, belongsToProject))
g.add((restriction1, OWL.cardinality, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((restriction1, OWL.allValuesFrom, ex.Project))

# MoveJob은 정확히 1개의 Project, Block, WorkingStage를 가짐
restriction2 = BNode()
g.add((ex.MoveJob, RDFS.subClassOf, restriction2))
g.add((restriction2, RDF.type, OWL.Restriction))
g.add((restriction2, OWL.onProperty, jobProject))
g.add((restriction2, OWL.cardinality, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((restriction2, OWL.allValuesFrom, ex.Project))

restriction3 = BNode()
g.add((ex.MoveJob, RDFS.subClassOf, restriction3))
g.add((restriction3, RDF.type, OWL.Restriction))
g.add((restriction3, OWL.onProperty, jobBlock))
g.add((restriction3, OWL.cardinality, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((restriction3, OWL.allValuesFrom, ex.Block))

restriction4 = BNode()
g.add((ex.MoveJob, RDFS.subClassOf, restriction4))
g.add((restriction4, RDF.type, OWL.Restriction))
g.add((restriction4, OWL.onProperty, jobStage))
g.add((restriction4, OWL.cardinality, Literal(1, datatype=XSD.nonNegativeInteger)))
g.add((restriction4, OWL.allValuesFrom, ex.WorkingStage))

print("Restriction 설정 완료")

## 3. 데이터 로드 및 변환

In [6]:
# CSV 데이터 로드 예시
df = pd.read_csv('../data/main/ai4i2020/ai4i2020.csv')
print(df.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    1     M14860    M                298.1                    308.6   
1    2     L47181    L                298.2                    308.7   
2    3     L47182    L                298.1                    308.5   
3    4     L47183    L                298.2                    308.6   
4    5     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1551         42.8                0                0    0   
1                    1408         46.3                3                0    0   
2                    1498         49.4                5                0    0   
3                    1433         39.5                7                0    0   
4                    1408         40.0                9                0    0   

   HDF  PWF  OSF  RNF  
0    0    0    0    0  
1    0    0    0    0  
2    0  

## 4. 온톨로지 저장

In [ ]:
# TTL 파일로 저장
# g.serialize("ontology/manufacturing.ttl", format="turtle")
# print("Ontology saved!")

## 5. SPARQL 쿼리

In [ ]:
# SPARQL 쿼리 예시
query = """
PREFIX ex: <http://example.org/ontology#>

SELECT ?s ?p ?o
WHERE {
    ?s ?p ?o .
}
LIMIT 10
"""

# results = g.query(query)
# for row in results:
#     print(row)